# CC2017 – Deep Learning · Proyecto Final
## 05 · Evaluación comparativa en el test oficial

Se cargan los checkpoints de los notebooks 02 y 03 y se evalúan sobre las 12,630 imágenes del test oficial de GTSRB, que ningún modelo vio durante el entrenamiento ni la selección de hiperparámetros.

## Preparación · Librerías, reproducibilidad y dispositivo

In [ ]:
import sys, random, time
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
OUT = ROOT / 'outputs'; OUT.mkdir(exist_ok=True)

# FAST_DEV_RUN=True recorre pocos lotes por época para comprobar que todo corre; sus checkpoints llevan prefijo dev_.
FAST_DEV_RUN = False
MAX_BATCHES = 5 if FAST_DEV_RUN else None
PREFIX = 'dev_' if FAST_DEV_RUN else ''
NUM_WORKERS = 2
print('Dispositivo:', device, '| FAST_DEV_RUN:', FAST_DEV_RUN)

## 1. Datos y modelos

In [ ]:
from src.data import download_gtsrb, load_tables, split_train_val, GTSRBDataset, get_transforms, CLASS_NAMES, NUM_CLASSES

DATA_ROOT = download_gtsrb()
train_full, test_df = load_tables(DATA_ROOT)
train_df, val_df = split_train_val(train_full, val_size=0.15, seed=SEED)
print(f'Train: {len(train_df):,} | Val: {len(val_df):,} | Test: {len(test_df):,}')

In [ ]:
from src.models import TrafficCNN, build_mobilenet_v2

def test_loader(size, imagenet_norm):
    ds = GTSRBDataset(test_df, DATA_ROOT, get_transforms(size, imagenet_norm=imagenet_norm))
    return DataLoader(ds, 128, num_workers=NUM_WORKERS)

def load(model, name):
    ckpt = torch.load(OUT / f'{PREFIX}{name}.pt', map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model']); return model.to(device).eval()

candidates = [
    ('CNN desde cero', lambda: TrafficCNN(), 'cnn_scratch', 48, False),
    ('MobileNetV2 congelado', lambda: build_mobilenet_v2(pretrained=False), 'mobilenet_frozen', 96, True),
    ('MobileNetV2 fine-tuning', lambda: build_mobilenet_v2(pretrained=False), 'mobilenet_finetuned', 96, True),
]
models_eval = [(name, load(make(), ck), size, norm) for name, make, ck, size, norm in candidates
               if (OUT / f'{PREFIX}{ck}.pt').exists()]
print('Modelos encontrados:', [m[0] for m in models_eval])

## 2. Métricas globales
Accuracy, precision, recall y F1. El promedio **macro** da el mismo peso a cada clase, así que penaliza fallar en las clases con pocos ejemplos. El **weighted** pondera por la cantidad de ejemplos de cada clase.

In [ ]:
from src.metrics import predict, summary_metrics, plot_confusion, top_confusions
preds, rows = {}, []
for name, model, size, norm in models_eval:
    loader = test_loader(size, norm)
    t0 = time.time()
    y, p, prob = predict(model, loader, device, MAX_BATCHES)
    preds[name] = (y, p, prob)
    rows.append({**summary_metrics(y, p, name), 'ms_por_imagen': 1000 * (time.time() - t0) / len(y)})
results = pd.DataFrame(rows).round(4)
results.to_csv(OUT / f'{PREFIX}metricas_test.csv', index=False)
results

## 3. Matrices de confusión
Matrices normalizadas por fila: la diagonal es el recall de cada clase. Las celdas fuera de la diagonal muestran con qué clase se confunde cada una.

In [ ]:
fig, axs = plt.subplots(1, len(preds), figsize=(9 * len(preds), 8))
axs = np.atleast_1d(axs)
for ax, (name, (y, p, _)) in zip(axs, preds.items()):
    plot_confusion(y, p, NUM_CLASSES, title=name, ax=ax)
plt.tight_layout(); plt.savefig(OUT / 'confusion_test.png', dpi=120); plt.show()

## 4. Confusiones más frecuentes

In [ ]:
for name, (y, p, _) in preds.items():
    print(name); display(top_confusions(y, p, CLASS_NAMES, k=8))

## 5. F1 por clase

In [ ]:
from sklearn.metrics import f1_score
f1_cls = pd.DataFrame({name: f1_score(y, p, average=None, labels=range(NUM_CLASSES), zero_division=0)
                       for name, (y, p, _) in preds.items()}, index=CLASS_NAMES)
ax = f1_cls.plot.bar(figsize=(16, 4), width=.8); ax.set(ylabel='F1', ylim=(0, 1.02), title='F1 por clase en test')
plt.tight_layout(); plt.savefig(OUT / 'f1_por_clase.png', dpi=120); plt.show()
f1_cls.min(axis=1).sort_values().head(10).to_frame('F1 mínimo entre modelos')

**Interpretación (completar después de ejecutar):** indique qué modelo gana en cada métrica y por cuánto, si la diferencia entre macro y weighted revela problemas con clases minoritarias, qué pares de clases se confunden (¿son señales visualmente parecidas, como los límites de velocidad?) y el costo en latencia de cada modelo.